In [0]:
from datetime import datetime
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, LongType, TimestampType

In [0]:
RAW = "/Volumes/workspace/bronze/raw_files"
LOG_TABLE = "workspace.bronze.ingestion_log"

In [0]:
# Control table
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {LOG_TABLE} (
    file_name     STRING,
    target_table  STRING,
    load_ts       TIMESTAMP,
    rows_read     BIGINT,
    rows_loaded   BIGINT,
    status        STRING,
    error_message STRING
)
""")

DataFrame[]

In [0]:
log_schema = StructType([
    StructField("file_name", StringType()),
    StructField("target_table", StringType()),
    StructField("load_ts", TimestampType()),
    StructField("rows_read", LongType()),
    StructField("rows_loaded", LongType()),
    StructField("status", StringType()),
    StructField("error_message", StringType()),
])

In [0]:
EXPECTED_COLS = {
    "workspace.bronze.clients": ["client_id", "nric", "full_name", "address", "region", "household_income", "household_size", "updated_date"],
    "workspace.bronze.schemes": ["scheme_id", "scheme_name", "category"],
    "workspace.bronze.assistance": ["case_id", "client_id", "scheme_id", "application_date","assistance_amount", "status"],
}

In [0]:
"""Returns true if this file has a SUCCESS entry in the control table."""
def already_loaded(file_name):
    return (
        spark.table(LOG_TABLE)
        .filter((F.col("file_name") == file_name) & (F.col("status") == "SUCCESS"))
        .count() > 0
    )

"""Create the entry in the control table"""
def log_run(file_name, target, rows_read, rows_loaded, status, err=None):
    row = [(file_name, target, datetime.now(), rows_read, rows_loaded, status, err)]
    spark.createDataFrame(row, schema=log_schema).write.mode("append").saveAsTable(LOG_TABLE)

""" Ingest a file into a table"""
def ingest(file_name, target_table):
    if already_loaded(file_name):
        print(f"SKIP: {file_name} already loaded")
        return
 
    rows_read = 0
    try:
        # Read everything as string, no cleaning, keep dirty rows.
        df = (
            spark.read
            .option("header", True)
            .option("inferSchema", False)
            .option("multiLine", True)
            .option("escape", '"') 
            .csv(f"{RAW}/{file_name}")
        )
        rows_read = df.count()
 
        # If schema does not match expectations, raise an error
        expected, actual = set(EXPECTED_COLS[target_table]), set(df.columns)
        if expected != actual:
            raise ValueError(f"Schema drift: missing={expected - actual}, extra={actual - expected}")
 
        df = (
            df.withColumn("ingest_ts", F.current_timestamp())
              .withColumn("source_file", F.lit(file_name))
        )
        df.write.mode("append").saveAsTable(target_table)
 
        log_run(file_name, target_table, rows_read, rows_read, "SUCCESS")
        print(f"OK: {file_name} -> {target_table} ({rows_read} rows)")
    except Exception as e:
        log_run(file_name, target_table, rows_read, 0, "FAILED", str(e)[:500])
        print(f"FAILED: {file_name}: {e}")

In [0]:
ingest("schemes.csv", "workspace.bronze.schemes")
ingest("clients_day1.csv", "workspace.bronze.clients")
ingest("assistance.csv", "workspace.bronze.assistance")

OK: schemes.csv -> workspace.bronze.schemes (8 rows)
OK: clients_day1.csv -> workspace.bronze.clients (1005 rows)
OK: assistance.csv -> workspace.bronze.assistance (2505 rows)


In [0]:
display(spark.table(LOG_TABLE))
display(spark.table("workspace.bronze.clients").groupBy("source_file").count())

file_name,target_table,load_ts,rows_read,rows_loaded,status,error_message
assistance.csv,workspace.bronze.assistance,2026-11-14T08:54:14.641Z,2505,2505,SUCCESS,null
clients_day1.csv,workspace.bronze.clients,2026-11-14T08:54:09.362Z,1005,1005,SUCCESS,null
schemes.csv,workspace.bronze.schemes,2026-11-14T08:54:04.284Z,8,8,SUCCESS,null


source_file,count
clients_day1.csv,1005


In [0]:
# Ingest the clients data of day 2
ingest("clients_day2.csv", "workspace.bronze.clients")


OK: clients_day2.csv -> workspace.bronze.clients (1020 rows)


In [0]:
display(spark.table(LOG_TABLE))
display(spark.table("workspace.bronze.clients").groupBy("source_file").count())

file_name,target_table,load_ts,rows_read,rows_loaded,status,error_message
assistance.csv,workspace.bronze.assistance,2026-11-14T08:54:14.641Z,2505,2505,SUCCESS,null
clients_day1.csv,workspace.bronze.clients,2026-11-14T08:54:09.362Z,1005,1005,SUCCESS,null
clients_day2.csv,workspace.bronze.clients,2026-11-14T08:57:33.688Z,1020,1020,SUCCESS,null
schemes.csv,workspace.bronze.schemes,2026-11-14T08:54:04.284Z,8,8,SUCCESS,null


source_file,count
clients_day2.csv,1020
clients_day1.csv,1005
